# 🎯 Ödev 2: Model Değerlendirme ve Optimizasyon

Bu ödevde Scikit-Learn içerisinde hazır bulunan ünlü bir **Meme Kanseri (Breast Cancer)** veri setini kullanacağız. Amacımız sadece tahmin yapmak değil, yapılan tahminin kalitesini doğru araçlarla (Accuracy, F1 Skoru) ölçebilmek ve modelin ayarlarını optimize etmektir (GridSearch).

> ⚠️ Aşağıdaki `init` hücresini çalıştırmayı unutmayın!

---

**English**

# Homework 2: Model Evaluation and Optimization
Use the sklearn Breast Cancer dataset. Evaluate predictions with accuracy and F1, then select the C parameter with GridSearch. Run init first.

[Colab’da aç / Open in Colab](https://colab.research.google.com/github/eliffozkan/GDGC-AI_ML_Course-26_27/blob/main/Module_2_Metrics/Module_2_Metrics_Homework.ipynb)

In [ ]:

def _evaluate_hw2(acc, f1, best_c, context=None):
    import numpy as np
    import warnings
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split, GridSearchCV
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score
    from sklearn.base import clone
    from sklearn.exceptions import ConvergenceWarning
    context = {} if context is None else context
    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
    models = [v for v in context.values() if isinstance(v, LogisticRegression)]
    matching_pairs = []
    for model in models:
        try:
            pred = model.predict(X_test)
            matching_pairs.append((accuracy_score(y_test, pred), f1_score(y_test, pred)))
        except Exception: pass
    def metric_ok(value, position):
        try:
            return bool(np.asarray(value).ndim == 0 and not isinstance(value, (bool, np.bool_)) and np.isfinite(value) and 0 <= value <= 1 and any(np.isclose(value, pair[position], rtol=0, atol=1e-8) for pair in matching_pairs))
        except Exception: return False
    ok_acc, ok_f1 = metric_ok(acc, 0), metric_ok(f1, 1)
    # Both metrics must describe the same trained model.
    if ok_acc and ok_f1:
        same_model = any(np.isclose(acc, a, atol=1e-8, rtol=0) and np.isclose(f1, f, atol=1e-8, rtol=0) for a, f in matching_pairs)
        ok_acc = ok_f1 = bool(same_model)
    results = [(35, ok_acc, "Görev 1: test accuracy hesabı / Task 1: test accuracy calculation"), (35, ok_f1, "Görev 2: test F1 hesabı (pozitif=1) / Task 2: test F1 calculation (positive=1)")]
    ok_grid = False
    for grid in [v for v in context.values() if isinstance(v, GridSearchCV)]:
        try:
            valid = isinstance(grid.estimator, LogisticRegression) and grid.cv == 3 and grid.scoring in [None, "accuracy"] and isinstance(grid.param_grid, dict) and set(grid.param_grid) == {"C"} and sorted(grid.param_grid["C"]) == [0.1, 1.0, 10.0]
            if not valid or isinstance(best_c, (bool, np.bool_)) or best_c != grid.best_params_["C"]: continue
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                reference = GridSearchCV(clone(grid.estimator), grid.param_grid, cv=3, scoring=grid.scoring).fit(X_train, y_train)
            # Validate the search results on the specified training split, not just membership in C candidates.
            if np.allclose(grid.cv_results_["mean_test_score"], reference.cv_results_["mean_test_score"], rtol=0, atol=1e-8) and grid.best_params_ == reference.best_params_:
                ok_grid = True
        except Exception: pass
    results.append((30, ok_grid, "Görev 3: eğitim setinde doğru GridSearch / Task 3: correct GridSearch on training data"))
    return results

def check_hw2(acc, f1, best_c):
    return _show_results(_evaluate_hw2(acc, f1, best_c, globals()))

def _show_results(results):
    score = 0
    print("--- Ödev Test Sonuçları / Homework Test Results ---")
    for points, ok, label in results:
        print(("✅ Başarılı / Passed: " if ok else "❌ Hatalı veya eksik / Incorrect or missing: ") + label)
        if ok: score += points
    print(f"Toplam Puan / Total Score: {score}/100")
    return score


## 🩺 Görev 1 & 2: Modeli Eğitme ve Metrik Hesaplama
Aşağıda sizin için veriyi yükledik ve Eğitim (Train) / Test olarak %80'e %20 oranında ayırdık. Kanser tümörünün iyi huylu mu kötü huylu mu olduğunu tahmin edeceğiz.

Sizden İstenen:
1. Bir `LogisticRegression` modeli eğitin. Sayısal yakınsama için aşağıdaki tekrarlanabilir ayarları kullanabilirsiniz.
2. Test seti (`X_test`) üzerinde tahminler yapın.
3. Gerçek değerler (`y_test`) ile tahminlerinizi karşılaştırarak `accuracy_score` ve `f1_score` metriklerini hesaplayın ve ilgili değişkenlere atayın.

Tekrarlanabilir örnek: `LogisticRegression(max_iter=1000, solver="liblinear", random_state=42)`. Eğitilmiş modeli bir değişkende saklayın; test gerçek tahminlerini yeniden hesaplar. F1 pozitif sınıfı 1 (iyi huylu), sınıf 0 kötü huyludur.

---

**English**

## Tasks 1 and 2: Train and evaluate
The data are split into 80% training and 20% test with random_state=42. Fit LogisticRegression on `X_train`/`y_train`, predict `X_test`, and compute accuracy_score and binary f1_score against `y_test`. Assign the values to `acc_skoru` and `f1_skoru`. Keep your fitted model in a variable; the checker recomputes its predictions. Use LogisticRegression(max_iter=1000, solver="liblinear", random_state=42) for a reproducible example. Class 1 is benign and is the positive class for F1; class 0 is malignant.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Veriyi yükleme ve bölme / Load and split the data
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# KODUNUZU BURAYA YAZIN / WRITE YOUR CODE HERE
acc_skoru = None
f1_skoru = None

print(f"Doğruluk (Accuracy): / Accuracy: {acc_skoru}")
print(f"F1 Skoru: / F1 score: {f1_skoru}")

## 🔎 Görev 3: Grid Search ile En İyi Parametreyi Bulma
Acaba Lojistik Regresyon içindeki hiperparametreleri (örneğin `C` parametresini) değiştirirsek başarı artar mı?

Sizden İstenen:
1. C parametresi için üç farklı değer deneyen bir GridSearchCV sistemi kurun: `{'C': [0.1, 1.0, 10.0]}`
2. `cv=3` ayarlayın.
3. Modeli eğittikten sonra en iyi C parametresini (grid_search nesnenizin `.best_params_` özelliğinden) çekerek `best_c` değişkenine atayın.

`scoring="accuracy"` kullanın; aramayı yalnızca `X_train`/`y_train` üzerinde eğitin. Eğitilmiş GridSearchCV nesnesini bir değişkende (ör. `grid_search`) saklayın. Test setinde arama yapmayın.

---

**English**

## Task 3: Find the best parameter with Grid Search
Create GridSearchCV with a LogisticRegression estimator, param_grid={"C": [0.1, 1.0, 10.0]}, cv=3 and scoring="accuracy". Fit only on `X_train`/`y_train`. Keep the fitted search object in a variable (for example `grid_search`). Assign its best_params_["C"] to `best_c`. Do not fit the search on the test set.

In [ ]:
from sklearn.model_selection import GridSearchCV

# KODUNUZU BURAYA YAZIN / WRITE YOUR CODE HERE
best_c = None

print("Modelin seçtiği en iyi C parametresi: / Best C selected by the model:", best_c)

## ✅ Kendini Test Et

---

**English**

## Self-check

In [ ]:
check_hw2(acc_skoru, f1_skoru, best_c)

**Teslim:** Hücreleri üstten alta çalıştırın, self-check sonucunu kontrol edin ve tamamladığınız `.ipynb` dosyasını mevcut ödev platformundaki modül alanına yükleyin. Platformdaki dosya adı ve son tarih talimatlarını izleyin.

**Submission:** Run the cells from top to bottom, review the self-check result, and upload the completed `.ipynb` file to the module area of the existing homework platform. Follow the platform’s file-name and deadline instructions.